# Data Preparation: Finetuning Datasets

Loads two datasets from OpenML — one for **classification** (BNP Paribas Cardif Claims Management, did=46625) and one for **regression** (Allstate Claims Severity, did=45046) — splits each 80/20 into train/score, and saves the results to Unity Catalog.

We use datasets with more than 100k sample to resemble real world examples where GPU memory becomes a bottleneck. 

**Prerequisites:** `common/config.py` defines the target catalog. The data will be populated in its 'default' schema. 

**Compute:** Serverless CPU, environment version 6

## Configuration

In [0]:
%pip install openml --quiet -q

In [0]:
dbutils.library.restartPython()

In [0]:
import importlib.util
import os
import sys

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import openml

# Make the repo-level common/ package importable.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
CONFIG_PATH = os.path.join(COMMON_PATH, "config.py")
SCHEMA = "default"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Could not find config.py at {CONFIG_PATH}")

spec = importlib.util.spec_from_file_location("project_config", CONFIG_PATH)
project_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(project_config)
CATALOG = project_config.CATALOG

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")

## Classification dataset: BNP Paribas Cardif Claims

In [0]:
# --- Classification: BNP Paribas Cardif Claims Management (OpenML did=46625) ---
dataset = openml.datasets.get_dataset(46625)
df, y, _, _ = dataset.get_data(dataset_format="dataframe")

# Fix dtypes that Arrow/Spark can't handle (categories → str, unsigned → signed)
df = df.astype(
    {c: str for c in df.select_dtypes("category").columns}
    | {c: "int32" for c in df.select_dtypes(include=[np.uint8, np.uint16, np.uint32, np.uint64]).columns}
)

print(f"Dataset: {dataset.name}")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")

# Keep only rows with a target value, then split (80/20)
df_labeled = df.dropna(subset=["target"])
print(f"Labeled rows: {len(df_labeled):,} (dropped {len(df) - len(df_labeled):,} with null target)")

df_train, df_score = train_test_split(df_labeled, test_size=0.2, random_state=42)

print(f"\nTrain: {df_train.shape[0]:,} rows")
print(f"Score: {df_score.shape[0]:,} rows")

# Save to Unity Catalog
TABLE_TRAIN = "finetuning_classification_train"
TABLE_SCORE = "finetuning_classification_score"

spark.createDataFrame(df_train).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_TRAIN}")
print(f"\n✓ Saved {df_train.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_TRAIN}")

spark.createDataFrame(df_score).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_SCORE}")
print(f"✓ Saved {df_score.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_SCORE}")

## Regression dataset: Allstate claims severity

In [0]:
# --- Regression: Allstate Claims Severity (OpenML did=45046, v4) ---
dataset = openml.datasets.get_dataset(45046)
df, y, _, _ = dataset.get_data(dataset_format="dataframe")

# Fix dtypes that Arrow/Spark can't handle (categories → str, unsigned → signed)
df = df.astype(
    {c: str for c in df.select_dtypes("category").columns}
    | {c: "int32" for c in df.select_dtypes(include=[np.uint8, np.uint16, np.uint32, np.uint64]).columns}
)

print(f"Dataset: {dataset.name}")
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print(f"Target: '{dataset.default_target_attribute}'")

# Split 80/20
df_train, df_score = train_test_split(df, test_size=0.2, random_state=42)

print(f"\nTrain: {df_train.shape[0]:,} rows")
print(f"Score: {df_score.shape[0]:,} rows")

# Save to Unity Catalog
TABLE_TRAIN = "finetuning_regression_train"
TABLE_SCORE = "finetuning_regression_score"

spark.createDataFrame(df_train).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_TRAIN}")
print(f"\n✓ Saved {df_train.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_TRAIN}")

spark.createDataFrame(df_score).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_SCORE}")
print(f"✓ Saved {df_score.shape[0]:,} rows to {CATALOG}.{SCHEMA}.{TABLE_SCORE}")

## Create Volumes to store finetuned models

This is not needed as MLFlow would handle the finetuned checkpoints, but we are storing them too in a volume to make explicit the reusability of finetuned models. 

In [0]:
# --- Create volumes for finetuned model artifacts ---
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.finetuned_classification_models")
print(f"✓ Volume: {CATALOG}.{SCHEMA}.finetuned_classification_models")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.finetuned_regression_models")
print(f"✓ Volume: {CATALOG}.{SCHEMA}.finetuned_regression_models")